In [30]:
import polars as pl
import datetime as dt
import os
from pathlib import Path

In [31]:
L2DATA_PATH = "/data/xujiayi/xjy/l2/"

In [32]:
def normalize_date(date: dt.date | dt.datetime | str) -> str:
    if isinstance(date, (dt.datetime, dt.date)):
        return date.strftime("%Y%m%d")
    return str(date).replace("-", "").replace(".", "").strip("/")

In [33]:
date = normalize_date('20260624')

filepath = Path(L2DATA_PATH)/"raw"/date

outpath = Path(L2DATA_PATH)/"proc"/date
outpath.mkdir(parents=True, exist_ok=True)

#### 深交所三表整理

In [5]:
szcj_merged = pl.read_parquet(filepath/'szcj.pq').filter(pl.col('MDStreamID')==11).drop(['MDStreamID','SecurityIDSource','LocalTime','SeqNo']).rename({
    'LastPx':'Price',
    'LastQty':'OrderQty'
}).with_columns([
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f"),
    pl.when(pl.col('BidApplSeqNum')>pl.col('OfferApplSeqNum')).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side')
])
szcj = szcj_merged.filter(pl.col('ExecType')==70).drop('ExecType')
szcancel = szcj_merged.filter(pl.col('ExecType')==52).drop('ExecType')

In [6]:
szcj.write_parquet(outpath/'szcj.pq',compression="gzip")
szcancel.write_parquet(outpath/'szcancel.pq',compression="gzip")

In [21]:
szwt = pl.read_parquet(filepath/'szwt.pq').filter(pl.col('MDStreamID')==11).drop(['MDStreamID','SecurityIDSource','LocalTime','SeqNo','OrdType'])
szwt = szwt.with_columns([
    pl.col('Side').replace(49,1).replace(50,-1).cast(pl.Int8),
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f")
])

In [22]:
szwt.write_parquet(outpath/'szwt.pq',compression="gzip")

#### 上交所三表整理

In [34]:
sh = pl.read_parquet(filepath/'sh.pq').drop(['TradeMoney','LocalTime','SeqNo','TickBSFlag']).rename({
    'BizIndex':'ApplSeqNum',
    'Channel':'ChannelNo',
    'TickTime':'TransactTime',
    'Qty':'OrderQty'
}).with_columns([
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f"),
])
sh

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Type,BuyOrderNO,SellOrderNO,Price,OrderQty
i64,i64,i64,time,str,i64,i64,f64,i64
1,1,751028,06:00:00.270,"""S""",0,0,0.0,0
2,1,751900,06:00:00.270,"""S""",0,0,0.0,0
3,1,751004,06:00:00.270,"""S""",0,0,0.0,0
4,1,751019,06:00:00.270,"""S""",0,0,0.0,0
5,1,751992,06:00:00.270,"""S""",0,0,0.0,0
…,…,…,…,…,…,…,…,…
46340824,5,688757,15:05:01.010,"""S""",0,0,0.0,0
46340825,5,603257,15:05:01.010,"""S""",0,0,0.0,0
46340826,5,563900,15:05:01.010,"""S""",0,0,0.0,0


In [41]:
shwt_added = sh.filter(pl.col('Type')=='A').drop('Type').with_columns([
    pl.when(pl.col('BuyOrderNO')!=0).then(pl.col('BuyOrderNO')).otherwise(pl.col('SellOrderNO')).alias('OrderNo'),
    pl.when(pl.col('BuyOrderNO')!=0).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side'),
]).drop(['BuyOrderNO','SellOrderNO'])
shwt_added 

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,OrderNo,Side
i64,i64,i64,time,f64,i64,i64,i32
663,1,603637,09:15:00.010,17.56,200,167,1
664,1,603637,09:15:00.250,21.33,100,1495,-1
665,1,603637,09:15:03.310,21.0,300,14934,-1
666,1,603637,09:15:03.350,20.39,700,15732,-1
667,1,603637,09:15:03.550,20.5,1200,19813,-1
…,…,…,…,…,…,…,…
45014564,3,603194,14:59:58.680,35.9,1100,27256813,1
45014565,3,603194,14:59:59.550,32.89,100,27258424,1
45014566,3,603194,14:59:59.610,35.12,1500,27258459,1


In [48]:
shcj = sh.filter(
    (pl.col('Type')=='T') | (pl.col('Type')=='D')
).join(
    shwt_added.select(['OrderNo','ApplSeqNum','SecurityID']),  # 获取买单的频道内索引
    left_on=['BuyOrderNO','SecurityID'],
    right_on=['OrderNo','SecurityID'],
    how='left',
    suffix='_buy'
).join(
    shwt_added.select(['OrderNo','ApplSeqNum','SecurityID']),  # 获取卖单的频道内索引
    left_on=['SellOrderNO','SecurityID'],
    right_on=['OrderNo','SecurityID'],
    how='left',
    suffix='_sell'
).rename({
    'ApplSeqNum_buy':'BidApplSeqNum',
    'ApplSeqNum_sell':'OfferApplSeqNum'
})
shcj

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Type,BuyOrderNO,SellOrderNO,Price,OrderQty,BidApplSeqNum,OfferApplSeqNum
i64,i64,i64,time,str,i64,i64,f64,i64,i64,i64
752,1,603637,09:15:20.120,"""D""",197408,0,19.4,900,751,null
761,1,603637,09:15:49.060,"""D""",0,213661,17.46,2700,null,759
877,1,603637,09:25:00.040,"""T""",368731,45418,19.33,400,847,684
878,1,603637,09:25:00.040,"""T""",368731,277909,19.33,400,847,785
879,1,603637,09:25:00.040,"""T""",368731,358795,19.33,1400,847,840
…,…,…,…,…,…,…,…,…,…,…
45014691,3,603194,15:00:02.270,"""T""",9584499,27168730,35.06,100,15403554,45014494
45014692,3,603194,15:00:02.270,"""T""",27144010,27168730,35.06,100,45014478,45014494
45014693,3,603194,15:00:02.270,"""T""",15679965,27168730,35.06,100,25592544,45014494


In [46]:
shcj = sh.filter(
    (pl.col('Type')=='T') | (pl.col('Type')=='D')
).join(
    shwt_added.select(['ChannelNo','OrderNo','ApplSeqNum','SecurityID']),  # 获取买单的频道内索引
    left_on=['ChannelNo','BuyOrderNO','SecurityID'],
    right_on=['ChannelNo','OrderNo','SecurityID'],
    how='left',
    suffix='_buy'
).join(
    shwt_added.select(['ChannelNo','OrderNo','ApplSeqNum','SecurityID']),  # 获取卖单的频道内索引
    left_on=['ChannelNo','SellOrderNO','SecurityID'],
    right_on=['ChannelNo','OrderNo','SecurityID'],
    how='left',
    suffix='_sell'
).rename({
    'ApplSeqNum_buy':'BidApplSeqNum',
    'ApplSeqNum_sell':'OfferApplSeqNum'
}).with_columns([
    pl.when(pl.col('Type')=='D').then(pl.col('BidApplSeqNum').fill_null(0)).otherwise(pl.col('BidApplSeqNum').fill_null(pl.col('ApplSeqNum'))),
    pl.when(pl.col('Type')=='D').then(pl.col('OfferApplSeqNum').fill_null(0)).otherwise(pl.col('OfferApplSeqNum').fill_null(pl.col('ApplSeqNum'))),
]).with_columns([                                               # 判断买卖方向
    pl.when(pl.col('BidApplSeqNum')>pl.col('OfferApplSeqNum')).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side')
]).drop(['BuyOrderNO','SellOrderNO'])
shcj

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Type,Price,OrderQty,BidApplSeqNum,OfferApplSeqNum,Side
i64,i64,i64,time,str,f64,i64,i64,i64,i32
752,1,603637,09:15:20.120,"""D""",19.4,900,751,0,1
761,1,603637,09:15:49.060,"""D""",17.46,2700,0,759,-1
877,1,603637,09:25:00.040,"""T""",19.33,400,847,684,1
878,1,603637,09:25:00.040,"""T""",19.33,400,847,785,1
879,1,603637,09:25:00.040,"""T""",19.33,1400,847,840,1
…,…,…,…,…,…,…,…,…,…
45014691,3,603194,15:00:02.270,"""T""",35.06,100,15403554,45014494,-1
45014692,3,603194,15:00:02.270,"""T""",35.06,100,45014478,45014494,-1
45014693,3,603194,15:00:02.270,"""T""",35.06,100,25592544,45014494,-1


In [8]:
shcancel = shcj.filter(pl.col('Type')=='D').drop('Type')
shcancel

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,BidApplSeqNum,OfferApplSeqNum,Side
i64,i64,i64,time,f64,i64,i64,i64,i32
752,1,603637,09:15:20.120,19.4,900,751,0,1
761,1,603637,09:15:49.060,17.46,2700,0,759,-1
1038,1,688150,09:15:18.780,46.71,200,0,1009,-1
1054,1,688150,09:16:41.660,58.39,1000,940,0,1
1062,1,688150,09:17:26.730,57.2,200,988,0,1
…,…,…,…,…,…,…,…,…
43086953,6,588170,14:59:59.970,3.0,400,10763750,0,1
43086954,6,513770,14:59:59.970,0.333,200,0,34466654,-1
43086955,6,513770,14:59:59.980,0.333,200,0,34466674,-1


In [9]:
shcancel.write_parquet(outpath/'shcancel.pq',compression="gzip")

In [10]:
shcj = shcj.filter(pl.col('Type')=='T').drop('Type')
shcj

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,BidApplSeqNum,OfferApplSeqNum,Side
i64,i64,i64,time,f64,i64,i64,i64,i32
877,1,603637,09:25:00.040,19.33,400,847,684,1
878,1,603637,09:25:00.040,19.33,400,847,785,1
879,1,603637,09:25:00.040,19.33,1400,847,840,1
880,1,603637,09:25:00.040,19.33,100,847,841,1
881,1,603637,09:25:00.040,19.33,200,847,747,1
…,…,…,…,…,…,…,…,…
45014691,3,603194,15:00:02.270,35.06,100,15403554,45014494,-1
45014692,3,603194,15:00:02.270,35.06,100,45014478,45014494,-1
45014693,3,603194,15:00:02.270,35.06,100,25592544,45014494,-1


In [11]:
shcj.write_parquet(outpath/'shcj.pq',compression="gzip")

In [12]:
shwt_added = shwt_added.drop('OrderNo')
shwt_added

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,Side
i64,i64,i64,time,f64,i64,i32
663,1,603637,09:15:00.010,17.56,200,1
664,1,603637,09:15:00.250,21.33,100,-1
665,1,603637,09:15:03.310,21.0,300,-1
666,1,603637,09:15:03.350,20.39,700,-1
667,1,603637,09:15:03.550,20.5,1200,-1
…,…,…,…,…,…,…
45014564,3,603194,14:59:58.680,35.9,1100,1
45014565,3,603194,14:59:59.550,32.89,100,1
45014566,3,603194,14:59:59.610,35.12,1500,1


In [15]:
def RestoreOrder(wt, cj):
    
    cj = cj.sort(["ChannelNo", "ApplSeqNum", "SecurityID", "TransactTime"])

    # 剔除集合竞价
    cj_df = cj.filter(~((pl.col("TransactTime") < pl.time(9, 30, 0)) | (pl.col("TransactTime") >= pl.time(14, 57, 0))))

    # 1. 从成交表提取买卖双方订单并汇总
    cj_buy = cj_df.select([
        "ChannelNo", "BidApplSeqNum", "SecurityID", "OrderQty", "Price", "TransactTime",
    ]).rename({"BidApplSeqNum": "ApplSeqNum"}).with_columns(pl.lit(1).alias("Side"))

    cj_sell = cj_df.select([
        "ChannelNo", "OfferApplSeqNum", "SecurityID", "OrderQty", "Price", "TransactTime",
    ]).rename({"OfferApplSeqNum": "ApplSeqNum"}).with_columns(pl.lit(-1).alias("Side"))

    cj_all = pl.concat([cj_buy, cj_sell])
    cj_summary = cj_all.group_by(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]).agg([
        pl.sum("OrderQty"),
        pl.last("Price"),  # 一笔主动单可能同时吃掉多笔挂单
        pl.last("TransactTime")
    ])

    # 2. 使用反连接和半连接分离三种情况
    # 部分成交：同时存在于委托和成交（inner join）
    partial = wt.join(
        cj_summary.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side", "OrderQty"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="inner"
    ).with_columns([
        (pl.col("OrderQty") + pl.col("OrderQty_right")).alias("OrderQty"),
        pl.lit("主动部分成交").alias("OrderStatus")
    ]).drop("OrderQty_right")

    # 未成交：存在于委托但不存在于成交（anti join）
    untouched = wt.join(
        cj_summary.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="anti"
    ).with_columns(
        pl.lit("挂单被动成交").alias("OrderStatus")
    )
    untouched = untouched.select(partial.columns)

    # 完全成交：存在于成交但不存在于委托（anti join）
    new = cj_summary.join(
        wt.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="anti"
    ).with_columns([
        pl.lit("主动完全成交").alias("OrderStatus"),
    ])
    new = new.select(partial.columns)

    # 3. 合并所有订单
    init_order = pl.concat([partial, untouched, new])
    return init_order

In [16]:
shwt = RestoreOrder(shwt_added, shcj)
shwt

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,Side,OrderStatus
i64,i64,i64,time,f64,i64,i32,str
42366131,5,688110,14:34:26.070,171.39,954,1,"""主动部分成交"""
28483937,1,603065,13:09:01.540,18.8,13800,-1,"""主动部分成交"""
36346274,6,603191,14:06:40.950,16.93,1200,-1,"""主动部分成交"""
18669907,6,600397,10:31:01.060,26.1,200,-1,"""主动部分成交"""
1890620,4,688008,09:34:03.790,259.09,400,1,"""主动部分成交"""
…,…,…,…,…,…,…,…
31377558,2,603690,13:14:00.100,31.4,600,-1,"""主动完全成交"""
11858116,5,603019,09:55:47.520,87.2,100,1,"""主动完全成交"""
8234992,1,600030,09:46:34.340,28.46,300,-1,"""主动完全成交"""


In [17]:
shwt.drop('OrderStatus').write_parquet(outpath/'shwt.pq',compression="gzip")

#### 检验还原委托的正确性

In [18]:
shwt = pl.read_parquet(outpath/'shwt.pq')
shcj = pl.read_parquet(outpath/'shcj.pq')
shcancel = pl.read_parquet(outpath/'shcancel.pq')

In [24]:
szcj = pl.read_parquet(outpath/'szcj.pq')
szwt = pl.read_parquet(outpath/'szwt.pq')
szcancel = pl.read_parquet(outpath/'szcancel.pq')

In [25]:
def get_close_orderbook(
    shwt: pl.DataFrame,
    shcj: pl.DataFrame,
    shcd: pl.DataFrame,
    topn: int = 10,
):
    # 1. 委托订单表
    orders = (
        shwt
        .select([
            "ChannelNo",
            "SecurityID",
            "ApplSeqNum",
            "Price",
            "OrderQty",
            "Side",
        ])
    )

    cj = pl.concat([shcj, shcd])

    # 2. 买方订单成交扣减
    buy_filled = (
        cj
        .select([
            "ChannelNo",
            "SecurityID",
            pl.col("BidApplSeqNum").alias("ApplSeqNum"),
            pl.col("OrderQty").alias("DealQty"),
        ])
        .with_columns(pl.lit(1).alias('Side'))
    )

    # 3. 卖方订单成交扣减
    sell_filled = (
        cj
        .select([
            "ChannelNo",
            "SecurityID",
            pl.col("OfferApplSeqNum").alias("ApplSeqNum"),
            pl.col("OrderQty").alias("DealQty"),
        ])
        .with_columns(pl.lit(-1).alias('Side'))
    )

    # 4. 每张订单的累计成交数量
    filled = (
        pl.concat([buy_filled, sell_filled])
        .group_by(["ChannelNo", "SecurityID", "Side", "ApplSeqNum"])
        .agg(
            pl.col("DealQty").sum().alias("FilledQty")
        )
    )

    # 8. 计算每张订单剩余数量
    alive_orders = (
        orders
        .join(
            filled,
            on=["ChannelNo", "SecurityID", "Side", "ApplSeqNum"],
            how="left",
        )
        .with_columns([
            (
                pl.col("OrderQty")
                - pl.col("FilledQty").fill_null(0)
            ).alias("RemainQty")
        ])
        .filter(pl.col("RemainQty") > 0)
    )

    # 9. 聚合成价格档位
    price_level = (
        alive_orders
        .group_by(["SecurityID", "Side", "Price"])
        .agg(
            pl.col("RemainQty").sum().alias("Qty")
        )
    )

    # 10. 买盘：价格从高到低
    bid_book = (
        price_level
        .filter(pl.col("Side") == 1)
        .with_columns(
            pl.col("Price")
            .rank(method="dense", descending=True)
            .over("SecurityID")
            .cast(pl.Int32)
            .alias("Level")
        )
        .filter(pl.col("Level") <= topn)
        .sort(["SecurityID", "Level"])
        .rename({
            "Price": "BidPrice",
            "Qty": "BidQty",
        })
        .select(["SecurityID", "Level", "BidPrice", "BidQty"])
    )

    # 11. 卖盘：价格从低到高
    ask_book = (
        price_level
        .filter(pl.col("Side") == -1)
        .with_columns(
            pl.col("Price")
            .rank(method="dense", descending=False)
            .over("SecurityID")
            .cast(pl.Int32)
            .alias("Level")
        )
        .filter(pl.col("Level") <= topn)
        .sort(["SecurityID", "Level"])
        .rename({
            "Price": "AskPrice",
            "Qty": "AskQty",
        })
        .select(["SecurityID", "Level", "AskPrice", "AskQty"])
    )

    # 12. 合并买卖盘
    close_book = (
        bid_book
        .join(
            ask_book,
            on=["SecurityID", "Level"],
            how="full",
            coalesce=True,
        )
        .sort(["SecurityID", "Level"])
    )

    return close_book, alive_orders

In [26]:
close_book, alive_orders = get_close_orderbook(szwt, szcj, szcancel, topn=10)
close_book

SecurityID,Level,BidPrice,BidQty,AskPrice,AskQty
i64,i32,f64,i64,f64,i64
1,1,10.51,1016877,10.52,45600
1,2,10.5,1697700,10.53,205800
1,3,10.49,145100,10.54,177800
1,4,10.48,414900,10.55,283400
1,5,10.47,121100,10.56,85800
…,…,…,…,…,…
302132,6,57.87,100,58.06,16000
302132,7,57.85,800,58.07,100
302132,8,57.82,1100,58.08,700


In [27]:
close_book.filter(pl.col('SecurityID')==1)

SecurityID,Level,BidPrice,BidQty,AskPrice,AskQty
i64,i32,f64,i64,f64,i64
1,1,10.51,1016877,10.52,45600
1,2,10.5,1697700,10.53,205800
1,3,10.49,145100,10.54,177800
1,4,10.48,414900,10.55,283400
1,5,10.47,121100,10.56,85800
1,6,10.46,79000,10.57,159600
1,7,10.45,380000,10.58,143700
1,8,10.44,49900,10.59,128000
1,9,10.43,193100,10.6,119500


In [28]:
close_book, alive_orders = get_close_orderbook(shwt, shcj, shcancel, topn=10)
close_book

SecurityID,Level,BidPrice,BidQty,AskPrice,AskQty
i64,i32,f64,i64,f64,i64
501001,1,1.516,35886,1.499,2969
501001,2,1.515,100,1.5,49704
501001,3,1.512,7192,1.502,62013
501001,4,1.511,34596,1.503,38876
501001,5,1.51,82942,1.504,45561
…,…,…,…,…,…
900948,6,2.75,13200,2.699,100
900948,7,2.749,49500,2.7,800
900948,8,2.748,9300,2.701,7300


In [29]:
close_book.filter(pl.col('SecurityID')==600000)

SecurityID,Level,BidPrice,BidQty,AskPrice,AskQty
i64,i32,f64,i64,f64,i64
600000,1,9.26,106200,8.91,3545400
600000,2,9.25,481100,8.92,4264300
600000,3,9.24,564400,8.93,1391900
600000,4,9.23,348600,8.94,249200
600000,5,9.22,543601,8.95,1764200
600000,6,9.21,995500,8.96,591500
600000,7,9.2,1072856,8.97,3329200
600000,8,9.19,576100,8.98,1119200
600000,9,9.18,762100,8.99,3316786
